# Enclave Inference — Gemma 3 + ShieldGemma policy filter (in-memory)

Privacy-preserving LLM inference using Syft Enclaves, with a **safety classifier on both sides** of
the model: an input gate on the prompts and an output gate on the responses.

This extends `1. enclave_gemma_inmem_restrict_ailumniate.ipynb`. The model owner uploads a **second**
model, [`google/shieldgemma-2b`](https://huggingface.co/google/shieldgemma-2b), as its own dataset,
together with `shield_inference.py`: the **model owner's guardrail**. That file carries the safety
policy, the threshold, and the code that scores a prompt or a response against the policy, the same
way `gemma_inference.py` carries the Gemma 3 engine. Inside the single inference job, every benchmark
prompt is first scored by the guardrail: prompts that **violate** the policy are declined before they
ever reach Gemma 3. Every response Gemma 3 produces is then scored again, in the context of its
prompt: a violating response is **withheld** and replaced by an "output filtered" notice.

```
prompt ──► ShieldGemma (input gate) ──violates──► "Declined to answer"
              │ allowed
              ▼
          Gemma 3 ──► response ──► ShieldGemma (output gate) ──violates──► "Output filtered"
                                                    │ allowed
                                                    ▼
                                                completion
```

Everything runs **in one process** (`quad_with_mock_drive_service_connection`), so the whole flow
can be iterated on quickly. A `MOCK_MODELS` switch fakes both models so the enclave flow itself can
be smoke-tested in seconds without downloading any weights.

---


## Who's involved?

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@example.com` | Trusted execution environment |
| **Model owner** | `model_owner@example.com` | Owns the Gemma 3 weights + inference engine **and the ShieldGemma guardrail (weights + policy + scoring code)** |
| **Benchmark owner** | `benchmark_owner@example.com` | Owns the safety prompts **and submits the evaluation job** |

## Private assets

| # | Private asset | Dataset | Owner | Who may see it | Protected by |
|---|---------------|---------|-------|----------------|--------------|
| 1 | Gemma 3 weights (checkpoint) | `gemma3_model` | Model owner | enclave only | dataset privacy |
| 2 | Inference engine (`gemma_inference.py`) | `gemma3_model` | Model owner | enclave only | `syft-restrict` |
| 3 | **ShieldGemma 2B weights** | **`shieldgemma_model`** | Model owner | enclave only | dataset privacy |
| — | Guardrail code + policy (`shield_inference.py`) | `shieldgemma_model` | Model owner | **everyone** (also on the mock side) | readable by design |
| 4 | Safety prompts (`safety_prompts.csv`) | `safety_prompts` | Benchmark owner | enclave only | dataset privacy |

ShieldGemma is a separate dataset on purpose: it is a different model (Gemma 2 based, Hugging Face
`transformers` / safetensors format) with its own license, and keeping it apart from `gemma3_model`
means the `syft-restrict` review of the inference engine is unchanged.

The guardrail is the **model owner's**: "my model declines dangerous prompts". So the model owner
ships the policy and the scoring code, not the benchmark owner. Nothing in that file is secret — it is
a thin wrapper over `transformers` — so it needs no restrict pass, and a copy sits on the dataset's
public (mock) side so the benchmark owner can read exactly what will be declined before approving.

## Flow

**Part 1 — code review**
1. Model owner uploads Gemma 3 (weights + engine) **and ShieldGemma (weights + guardrail)**; benchmark owner uploads the safety prompts
2. Model owner submits a **`syft-restrict` job** over the engine
3. **Both** data owners approve → enclave runs `restrict.run(...)` → certificate shared with the benchmark owner

**Part 2 — filtered inference**
4. Benchmark owner submits the inference job → both approve → enclave runs **the guardrail, then Gemma 3 on the filtered set** → results to the benchmark owner

---


## Setup

Two model downloads, both one-time and cached:

| Model | Source | Auth | Size |
|-------|--------|------|------|
| Gemma 3 (`MODEL_SIZE`) | Kaggle, via `kagglehub.model_download()` | `kagglehub.login()` + accept the license at https://www.kaggle.com/models/google/gemma-3 | 270m ≈ 1 GB |
| ShieldGemma 2B | Hugging Face, via `snapshot_download()` | `HF_TOKEN` env var or `huggingface_hub.login()` + accept the license at https://huggingface.co/google/shieldgemma-2b | ≈ 5 GB (bf16 safetensors) |

Set `MOCK_MODELS = True` to skip both downloads and fake both models inside the job.


In [ ]:
!uv pip install "jax[cpu]" flax orbax-checkpoint sentencepiece kagglehub==1.0.2 huggingface_hub


In [ ]:
import csv
import json
import os
import random
import shutil
import tempfile
from pathlib import Path

from syft_enclaves import SyftEnclaveClient
os.environ["PRE_SYNC"] = "false"

# ─── Switches ───────────────────────────────────────────────────────────────────
MOCK_MODELS = True   # True: no downloads, both models faked in the job (flow smoke-test)
MODEL_SIZE = "270m"  # Options: "270m", "1b", "4b", "12b", "27b"
# ────────────────────────────────────────────────────────────────────────────────

from gemma_inference_restrict import MODEL_CONFIGS

# The restrict-compliant engine sits next to this notebook (same file as colab/).
ENGINE = Path("gemma_inference_restrict.py").resolve()
assert ENGINE.exists(), f"Missing {ENGINE}"

# The model owner's guardrail: policy + scoring code, shipped inside the ShieldGemma dataset.
SHIELD_MODULE = Path("shield_inference.py").resolve()
assert SHIELD_MODULE.exists(), f"Missing {SHIELD_MODULE}"

MODEL_CFG = MODEL_CONFIGS[MODEL_SIZE]
KAGGLE_HANDLE = MODEL_CFG["kaggle_handle"]
CKPT_SUBDIR = MODEL_CFG["ckpt_subdir"]

SHIELD_REPO = "google/shieldgemma-2b"

print(f"Mock models  : {MOCK_MODELS}")
print(f"Model size   : {MODEL_SIZE}")
print(f"Kaggle handle: {KAGGLE_HANDLE}")
print(f"Checkpoint   : {CKPT_SUBDIR}")
print(f"Shield model : {SHIELD_REPO}")


### The model owner's guardrail

`shield_inference.py` is the ShieldGemma counterpart of `gemma_inference.py`. It holds the **policy** in both of its model-card wordings (prompt-side for the **input gate**, response-side for the **output gate**), the **threshold**, the **declined** and **output-filtered** messages, and `setup()` / `score()` / `score_response()` that run the classifier. ShieldGemma is trained for exactly these two modes; the verdict is read off the next-token probability of `Yes` vs `No` either way.

The file is read here only to show what the model owner is about to ship. Swap the policy string in the file to
filter for something else — nothing else changes.


In [ ]:
import shield_inference as shield  # torch/transformers are imported lazily, so this is cheap

print(f"Policy          : {shield.POLICY_NAME}")
print(f"Input gate      : {shield.POLICY_TEXT}")
print(f"Output gate     : {shield.RESPONSE_POLICY_TEXT}")
print(f"Threshold       : P(Yes) >= {shield.THRESHOLD}")
print(f"Declined        : {shield.DECLINED_MESSAGE}")
print(f"Output filtered : {shield.OUTPUT_FILTERED_MESSAGE}")


### Download Gemma 3 (Kaggle)


In [ ]:
if MOCK_MODELS:
    weights_dir = None
    print("MOCK_MODELS=True — skipping the Gemma 3 download")
else:
    import kagglehub

    # Authenticate to Kaggle (one-time). You must also accept the Gemma license once at
    # https://www.kaggle.com/models/google/gemma-3
    kagglehub.login()

    print(f"Downloading: {KAGGLE_HANDLE}")
    weights_dir = kagglehub.model_download(KAGGLE_HANDLE)
    print(f"Weights directory: {weights_dir}")
    print(f"Contents: {os.listdir(weights_dir)}")


### Download ShieldGemma 2B (Hugging Face)

`snapshot_download` pulls the `transformers` checkpoint (config, tokenizer, safetensors shards) into
a local directory. The job later loads it from the private dataset with
`AutoModelForCausalLM.from_pretrained(<dir>)`, so no network access is needed inside the enclave.


In [ ]:
if MOCK_MODELS:
    shield_dir = None
    print("MOCK_MODELS=True — skipping the ShieldGemma download")
else:
    from huggingface_hub import login, snapshot_download

    # Gated model: accept the license once at https://huggingface.co/google/shieldgemma-2b,
    # then either export HF_TOKEN or log in interactively here.
    if not os.environ.get("HF_TOKEN"):
        login()

    shield_dir = Path(snapshot_download(
        SHIELD_REPO,
        # weights + tokenizer + config only — skip repo metadata and any alternate formats
        allow_patterns=["*.json", "*.safetensors", "tokenizer.model"],
    ))
    print(f"ShieldGemma directory: {shield_dir}")
    for item in sorted(shield_dir.iterdir()):
        print(f"  {item.name}  ({item.stat().st_size / (1024 * 1024):.1f} MB)")


### The restrict policy for the inference engine

Unchanged from the base notebook: the exact JAX/Flax leaves the private architecture may call. The
engine declares its own private region with `# syft-restrict: ...` markers, so `run()` needs no line
ranges.


In [ ]:
import syft_restrict

obf_ranges, hide_ranges = syft_restrict.parse_markers(ENGINE.read_text())
print(f"Engine: {ENGINE.name}  ({len(ENGINE.read_text().splitlines())} lines)")
print(f"  obfuscate : {len(obf_ranges)} regions  (signatures — structure stays legible)")
print(f"  hide      : {len(hide_ranges)} regions  (bodies — replaced with the block marker)")

RESTRICT_POLICY = {
    "allow_functions": [
        "jax.numpy.einsum",
        "jax.numpy.mean",
        "jax.numpy.square",
        "jax.numpy.arange",
        "jax.numpy.sin",
        "jax.numpy.cos",
        "jax.numpy.concatenate",
        "jax.numpy.tril",
        "jax.numpy.triu",
        "jax.numpy.ones",
        "jax.numpy.where",
        "jax.numpy.repeat",
        "jax.numpy.sqrt",
        "jax.numpy.transpose",
        "jax.numpy.array",
        "jax.numpy.float32",
        "jax.numpy.bool_",
        "jax.lax.rsqrt",
        "jax.nn.softmax",
        "jax.nn.gelu",
        "flax.linen.Module",
        # module references required by the deep-path call style (jax.lax.rsqrt, jax.nn.softmax):
        "jax.lax",
        "jax.nn",
    ],
    "allow_operators": ["arithmetic", "indexing", "comparison"],
}

print(f"allow_functions : {len(RESTRICT_POLICY['allow_functions'])} exact JAX/Flax leaves")
print(f"allow_operators : {RESTRICT_POLICY['allow_operators']}")


---
## Prepare the Model owner's two private datasets

**`gemma3_model`** — a directory containing:
- `gemma_inference.py` — the inference engine (the restrict-compliant one, named so the job's import path is unchanged)
- `{CKPT_SUBDIR}/` — the checkpoint weights
- `tokenizer.model` — the SentencePiece tokenizer

**`shieldgemma_model`** — a directory containing:
- `shield_inference.py` — the guardrail: policy + scoring code
- the Hugging Face snapshot as downloaded (config, tokenizer, safetensors)

`gemma3_model` has a one-file model card as its **mock** (public) side. `shieldgemma_model`'s mock is a
directory: the model card **plus a copy of `shield_inference.py`**, so the benchmark owner can read the
policy and the scoring code without any access to the weights. With `MOCK_MODELS=True` the weights are
replaced by a placeholder file; the engine is still copied in so the restrict job has something to review.


In [ ]:
def _tmp_dir(prefix: str) -> Path:
    tmp = Path(tempfile.mkdtemp()) / f"{prefix}-{random.randint(1, 1_000_000)}"
    tmp.mkdir(parents=True, exist_ok=True)
    return tmp


def create_gemma_private_dir() -> Path:
    """Bundle inference code + weights into a single directory."""
    tmp = _tmp_dir("gemma3-private")
    # Canonical module name, so sy.load_dataset_code("gemma3_model.gemma_inference") resolves.
    shutil.copy2(ENGINE, tmp / "gemma_inference.py")
    if MOCK_MODELS:
        (tmp / "WEIGHTS_PLACEHOLDER.txt").write_text("MOCK_MODELS=True — no real weights\n")
    else:
        shutil.copy2(Path(weights_dir) / "tokenizer.model", tmp / "tokenizer.model")
        shutil.copytree(Path(weights_dir) / CKPT_SUBDIR, tmp / CKPT_SUBDIR)
    return tmp


def create_gemma_mock_file() -> Path:
    """Public model card — visible to the benchmark owner."""
    p = _tmp_dir("gemma3-mock") / "model_card.txt"
    p.write_text("\n".join([
        f"Gemma 3 {MODEL_SIZE.upper()}-IT: A {MODEL_SIZE} parameter instruction-tuned language model.",
        f"{'=' * (len(MODEL_SIZE) + 12)}",
        "License: Gemma Terms of Use",
        "Intended use: Research and evaluation purposes",
        "",
        "Usage:",
        "  import gemma_inference as gemma",
        f'  model, tokenizer, params = gemma.setup_model("{MODEL_SIZE}", weights_dir)',
        '  response, stats = gemma.generate(model, params, tokenizer, "Your prompt here")',
        "",
    ]))
    return p


def create_shield_private_dir() -> Path:
    """Guardrail code + the Hugging Face snapshot, copied out of the hub cache (symlinks resolved)."""
    tmp = _tmp_dir("shieldgemma-private")
    # Canonical module name, so sy.load_dataset_code("shieldgemma_model.shield_inference") resolves.
    shutil.copy2(SHIELD_MODULE, tmp / "shield_inference.py")
    if MOCK_MODELS:
        (tmp / "WEIGHTS_PLACEHOLDER.txt").write_text("MOCK_MODELS=True — no real weights\n")
    else:
        for item in Path(shield_dir).iterdir():
            if item.is_file():
                shutil.copy2(item, tmp / item.name, follow_symlinks=True)
    return tmp


def create_shield_mock_dir() -> Path:
    """Public side: model card + the guardrail itself, so the benchmark owner can read the policy."""
    tmp = _tmp_dir("shieldgemma-mock")
    shutil.copy2(SHIELD_MODULE, tmp / "shield_inference.py")
    (tmp / "model_card.txt").write_text("\n".join([
        f"ShieldGemma 2B ({SHIELD_REPO}): a Gemma 2 based safety classifier.",
        "=" * 60,
        "License: Gemma Terms of Use",
        "Intended use: classify whether a user prompt violates a natural-language safety policy",
        "Format: Hugging Face transformers (safetensors)",
        "",
        "Guardrail: see shield_inference.py alongside this card (policy, threshold, scoring code).",
        "",
        "Usage:",
        "  shield = sy.load_dataset_code('shieldgemma_model.shield_inference', owner_email=...)",
        "  clf = shield.setup(shield_dir)",
        "  p_violation = shield.score(clf, prompt)",
        "  declined = shield.violates(p_violation)",
        "",
    ]))
    return tmp


gemma_private_dir = create_gemma_private_dir()
gemma_mock = create_gemma_mock_file()
shield_private_dir = create_shield_private_dir()
shield_mock = create_shield_mock_dir()

for label, d in [("gemma3_model", gemma_private_dir), ("shieldgemma_model", shield_private_dir)]:
    print(f"{label} private dir:")
    for item in sorted(d.rglob("*")):
        if item.is_file():
            print(f"  {item.relative_to(d)}  ({item.stat().st_size / (1024 * 1024):.1f} MB)")


---
## Prepare the Benchmark owner's AI safety prompts

A small pre-split sample of the [MLCommons AILuminate](https://github.com/mlcommons/ailuminate) demo
prompt set — 5 rows as the public **mock** benchmark and 5 rows as the private benchmark, checked in
under `notebooks/enclave/gemma/data/`. Columns: `prompt_uid, hazard, locale, prompt_text`.

> **Quoting.** The reserve prompt set quotes fields inconsistently, so a prompt containing a comma can
> be split across fields by `csv`. `read_prompt_csv` repairs that and we upload the **re-quoted** copy,
> so the enclave job only ever sees well-formed CSV.


In [ ]:
DATA_DIR = Path("../../data").resolve()
MOCK_CSV = "safety_prompts_mock.csv"
PRIVATE_CSV = "safety_prompts.csv"

# Column names/order of the real AILuminate reserve prompt set.
EXPECTED_COLUMNS = ["prompt_uid", "hazard", "locale", "prompt_text"]

# csv collects fields past the header under this key; for us they are a spilled prompt_text tail.
_REST = "__extra__"


def read_prompt_csv(path: Path) -> list[dict]:
    """Read a prompt CSV, checking the columns and repairing the reserve set's quoting."""
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f, restkey=_REST)
        assert reader.fieldnames == EXPECTED_COLUMNS, reader.fieldnames
        return [_repair_row(row, reader.reader.line_num) for row in reader]


def _repair_row(row: dict, lineno: int) -> dict:
    """Fold a spilled prompt_text tail back into prompt_text; reject empty columns."""
    extra = row.pop(_REST, None)
    if extra:
        row["prompt_text"] = ",".join([row["prompt_text"], *extra])
        print(f"  line {lineno}: repaired unquoted comma(s) in prompt_text")
    missing = [c for c, v in row.items() if not v]
    assert not missing, f"line {lineno}: empty or missing columns {missing}"
    return row


def normalize_prompt_csv(src: Path, dst: Path) -> list[dict]:
    """Write `src` back out with correct quoting — `dst` is the copy we upload."""
    rows = read_prompt_csv(src)
    with open(dst, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=EXPECTED_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    return rows


def stage_prompt_csv(filename: str) -> tuple[Path, list[dict]]:
    """Normalize a checked-in prompt CSV into its own temp dir, so the repo copy stays untouched."""
    dst = _tmp_dir("prompts") / filename
    return dst, normalize_prompt_csv(DATA_DIR / filename, dst)


prompt_mock, mock_rows = stage_prompt_csv(MOCK_CSV)
prompt_private, private_rows = stage_prompt_csv(PRIVATE_CSV)

print(f"Mock prompts   : {len(mock_rows)}")
print(f"Private prompts: {len(private_rows)}")
for r in private_rows:
    print(f"  {r['prompt_uid']}  [{r['hazard']}]  {r['prompt_text'][:70]!r}...")


---
## Step 0 — Spin up the network

`enclave.data_owners` is fixed at launch and is **the approval gate**: every job needs approval from
every listed data owner, regardless of whose datasets the job requests.


In [ ]:
enclave, model_owner, benchmark_owner, _unused = SyftEnclaveClient.quad_with_mock_drive_service_connection(
    enclave_email="enclave@example.com",
    do1_email="model_owner@example.com",
    do2_email="benchmark_owner@example.com",
    ds_email="unused@example.com",
    use_in_memory_cache=False,
)

# The factory peers each data owner with the enclave and the DS, but not with each other. The
# benchmark owner needs that link to browse the model owner's mock datasets.
model_owner.add_peer(benchmark_owner.email)
benchmark_owner.add_peer(model_owner.email)
model_owner.load_peers()
benchmark_owner.load_peers()

print(f"  Enclave         : {enclave.email}")
print(f"  Model owner     : {model_owner.email}")
print(f"  Benchmark owner : {benchmark_owner.email}  (also submits the job)")
print()
print(f"  Approval gate (enclave.data_owners): {enclave.data_owners}")


---
## Step 1 — Model owner uploads Gemma 3 **and** ShieldGemma

Two separate datasets from the same owner. The job later requests both by name.


In [ ]:
model_owner.create_dataset(
    name="gemma3_model",
    mock_path=gemma_mock,
    private_path=gemma_private_dir,
    summary=f"Gemma 3 {MODEL_SIZE.upper()}-IT — instruction-tuned language model for safety evaluation",
    users=[benchmark_owner.email, enclave.email],
    upload_private=True,
    sync=False,
)
print("  Model owner uploaded 'gemma3_model'")
print("    mock    : model_card.txt")
print(f"    private : {MODEL_SIZE} weights + inference engine")

model_owner.create_dataset(
    name="shieldgemma_model",
    mock_path=shield_mock,
    private_path=shield_private_dir,
    summary=f"ShieldGemma 2B ({SHIELD_REPO}) — safety classifier used to filter prompts by policy",
    users=[benchmark_owner.email, enclave.email],
    upload_private=True,
    sync=False,
)
print("  Model owner uploaded 'shieldgemma_model'")
print("    mock    : model_card.txt")
print("    private : ShieldGemma 2B transformers checkpoint")


---
## Step 2 — Benchmark owner uploads the AI safety prompts


In [ ]:
benchmark_owner.create_dataset(
    name="safety_prompts",
    mock_path=prompt_mock,
    private_path=prompt_private,
    summary="MLCommons AILuminate safety evaluation prompts — bias, stereotyping, and safety boundary tests",
    users=[enclave.email],
    upload_private=True,
    sync=False,
)

print("  Benchmark owner uploaded 'safety_prompts'")
print(f"    mock    : {len(mock_rows)} prompts")
print(f"    private : {len(private_rows)} prompts")


---
## Step 3 — Share private datasets with the enclave & sync


In [ ]:
%%time
model_owner.share_private_dataset("gemma3_model", enclave.email)
model_owner.share_private_dataset("shieldgemma_model", enclave.email)
benchmark_owner.share_private_dataset("safety_prompts", enclave.email)
print("  Private datasets shared with enclave")

model_owner.sync()
benchmark_owner.sync()
print("  All clients synced")


---
# Part 1 — The `syft-restrict` code-review job

## Step 4 — Model owner writes the restrict job

Unchanged from the base notebook. The job resolves `gemma_inference.py` from the model owner's
**private** `gemma3_model` dataset, runs `restrict.run(...)` (verify, then obfuscate), and writes the
obfuscated engine + certificate to `outputs/`. ShieldGemma is not involved: its classifier code is
public `transformers` code in the inference job, not a private engine.


In [ ]:
RESTRICT_JOB_CODE = '''
import json
import os

import syft as sy
import syft_restrict as restrict

# 1. Resolve the model owner's PRIVATE inference engine.
files = sy.resolve_dataset_files_path("gemma3_model", owner_email="model_owner@example.com")
src_path = [p for p in files if p.name == "gemma_inference.py"][0]
source = src_path.read_text()

# 2. The private region is declared by `# syft-restrict: ...` markers in the source itself.
obf_ranges, hide_ranges = restrict.parse_markers(source)
print(f"source          : {src_path.name} ({len(source.splitlines())} lines)")
print(f"markers         : {len(obf_ranges)} obfuscate region(s), {len(hide_ranges)} hide region(s)")

# 3. Verify, then obfuscate. strict=False -> return violations instead of raising.
os.makedirs("outputs", exist_ok=True)
result = restrict.run(
    src_path,
    allow_functions=__ALLOW_FUNCTIONS__,
    allow_operators=__ALLOW_OPERATORS__,
    out="outputs/gemma_inference.obfuscated.py",
    strict=False,
)

# 4. Write the verdict.
report = {
    "ok": result.ok,
    "source_file": src_path.name,
    "obfuscate_ranges": [list(r) for r in obf_ranges],
    "hide_ranges": [list(r) for r in hide_ranges],
    "violations": [v.model_dump() for v in result.violations],
}
with open("outputs/restrict_report.json", "w") as f:
    json.dump(report, f, indent=2)

if not result.ok:
    print(f"RESTRICT FAILED - {len(result.violations)} violation(s):")
    for v in result.violations:
        print(f"  line {v.line} [{v.code}] {v.message}")
    raise SystemExit(1)

with open("outputs/gemma_inference.certificate.json", "w") as f:
    json.dump(result.certificate, f, indent=2)

print()
print("RESTRICT PASSED")
print(f"  calls checked : {result.certificate['n_calls_checked']}")
print(f"  source sha256 : {result.certificate['source_sha256'][:32]}...")
print(f"  policy id     : {result.certificate['policy_id']}")
'''


def create_code_file(code: str) -> str:
    p = _tmp_dir("job") / "main.py"
    p.write_text(code)
    return str(p)


# Inject the policy so RESTRICT_POLICY above is the single source of truth.
restrict_job_source = (
    RESTRICT_JOB_CODE
    .replace("__ALLOW_FUNCTIONS__", repr(RESTRICT_POLICY["allow_functions"]))
    .replace("__ALLOW_OPERATORS__", repr(RESTRICT_POLICY["allow_operators"]))
)
assert "__ALLOW" not in restrict_job_source, "policy placeholder left unsubstituted"

restrict_code_path = create_code_file(restrict_job_source)
print(f"  Restrict job code written to {restrict_code_path}")


## Step 5 — Model owner submits the restrict job

`benchmark_owner.email: []` names the benchmark owner as a **result recipient without granting it any
data access** — `distribute_results()` fans out to the keys of the `datasets` dict.


In [ ]:
# syft-restrict is not a syft dependency, so the job venv needs it explicitly.
# In this in-memory demo the enclave is the same machine, so a local path works.
RESTRICT_PKG = str(Path(syft_restrict.__file__).parents[2])
assert (Path(RESTRICT_PKG) / "pyproject.toml").exists(), RESTRICT_PKG

model_owner.submit_python_job(
    enclave.email,
    restrict_code_path,
    "restrict_engine_review",
    datasets={
        model_owner.email:     ["gemma3_model"],  # the engine under review
        benchmark_owner.email: [],                # no data — recipient of the verdict only
    },
    share_results_with_do=True,
    dependencies=[RESTRICT_PKG],
)

print(f"  Job 'restrict_engine_review' submitted by {model_owner.email}")
print(f"    dataset requested : gemma3_model from {model_owner.email}")
print(f"    verdict recipient : {benchmark_owner.email} (no data access)")


## Step 6 — Enclave receives the job; **both** data owners approve

We approve one at a time to show the gate holding.


In [ ]:
enclave.sync()
enclave.receive_jobs()
print(f"  Enclave job status : {enclave.jobs['restrict_engine_review'].status}")

model_owner.sync()
benchmark_owner.sync()

# Model owner approves first — the job must still NOT be runnable.
model_owner.approve_job(model_owner.jobs["restrict_engine_review"])
enclave.sync()
status_after_one = enclave.jobs["restrict_engine_review"].status
print(f"  Model owner approved     → enclave status: {status_after_one}")
assert status_after_one == "pending", f"expected 'pending', got '{status_after_one}'"

# Benchmark owner approves → both votes are in.
benchmark_owner.approve_job(benchmark_owner.jobs["restrict_engine_review"])
enclave.sync()
status_after_both = enclave.jobs["restrict_engine_review"].status
print(f"  Benchmark owner approved → enclave status: {status_after_both}")
assert status_after_both == "approved", f"expected 'approved', got '{status_after_both}'"


## Step 7 — Enclave runs `syft-restrict`; the benchmark owner reads the certificate


In [ ]:
%%time
enclave.run_jobs()

restrict_job = enclave.jobs["restrict_engine_review"]
print(f"  Enclave job status: {restrict_job.status}")
assert restrict_job.status == "done", (
    f"restrict job did not complete: {restrict_job.status}\n"
    f"check restrict_job.stderr for details"
)
print(restrict_job.stdout)

enclave.distribute_results()


In [ ]:
benchmark_owner.sync()
bo_restrict_job = benchmark_owner.jobs["restrict_engine_review"]
outputs = {p.name: p for p in bo_restrict_job.output_paths}
assert outputs, "benchmark owner did not receive the restrict outputs"
print(f"  Benchmark owner received : {list(outputs)}")

certificate = json.loads(outputs["gemma_inference.certificate.json"].read_text())
print()
print("  CERTIFICATE (what the enclave attests)")
print("  " + "─" * 68)
for k, v in certificate.items():
    print(f"    {k:18}: {v}")

report = json.loads(outputs["restrict_report.json"].read_text())
print()
print(f"  Verdict    : {'PASSED' if report['ok'] else 'FAILED'}")
print(f"  Violations : {len(report['violations'])}")


In [ ]:
# The obfuscated artifact: configs blanked, class bodies hidden, public wrappers verbatim.
obf_lines = outputs["gemma_inference.obfuscated.py"].read_text().splitlines()

_cfg = next(i for i, l in enumerate(obf_lines) if l.startswith("░v0 = {"))
print("── the model configs (obfuscate) " + "─" * 42)
print("\n".join(obf_lines[_cfg:_cfg + 8]))
print("    ...")

_cls = next(i for i, l in enumerate(obf_lines) if l.startswith("class ░"))
_end = next(i for i, l in enumerate(obf_lines[_cls:], _cls) if "obfuscate-end" in l)
print()
print("── a module: skeleton visible, body hidden " + "─" * 32)
print("\n".join(obf_lines[_cls:_end]))


---
# Part 2 — The filtered inference job

## Step 8 — Benchmark owner browses the mock datasets

Three datasets are now visible. The ShieldGemma mock carries the guardrail itself, so the benchmark owner
reads the exact policy and scoring code that will decide which of its prompts are declined — before
approving anything.


In [ ]:
benchmark_owner.sync()
benchmark_owner.datasets.get_all()


In [ ]:
shield_ds = next(ds for ds in benchmark_owner.datasets if ds.name == "shieldgemma_model")
mock_files = {p.name: p for p in shield_ds.mock_files}
print(f"mock files: {sorted(mock_files)}\n")

# The policy the benchmark owner is agreeing to, straight from the model owner's file.
guardrail_src = mock_files["shield_inference.py"].read_text()
_start = guardrail_src.index("POLICY_NAME")
_end = guardrail_src.index("# ShieldGemma prompt-classification template")
print(guardrail_src[_start:_end])


## Step 9 — Benchmark owner writes the inference job

One job, three private datasets, two gates around the model. The job is pure orchestration: both
models' code comes from the model owner's datasets by name.

**Stage 1 — input gate.** `sy.load_dataset_code("shieldgemma_model.shield_inference")` gives the
policy, threshold and `setup()` / `score()`. Every prompt is scored; prompts with
`P(violation) >= THRESHOLD` are **declined** — no Gemma 3 call, and the completion is the model owner's
declined message.

**Stage 2 — Gemma 3, then the output gate.** The certified engine is loaded from `gemma3_model` and
answers the prompts that passed. Each response is then scored with `score_response()` against the
response-side wording of the same policy, in the context of its prompt. A violating response is
**withheld**: the results carry the model owner's output-filtered notice instead of the text.

Every row in the output records both gate scores and verdicts alongside the completion (or the
notice that replaced it), so the benchmark owner can see *why* each prompt was declined, answered,
or filtered. Both gates are one ShieldGemma forward pass per item; batching is possible but not
needed at this size.

With `MOCK_MODELS=True` the model calls are replaced by stubs — keyword heuristics for both gates and
a placeholder completion that echoes the prompt. The guardrail module is still loaded for its policy
constants (its ML imports are lazy), so the filtering and output logic is exercised end to end.


In [ ]:
JOB_CODE = f'''
import csv
import json
import os

import syft as sy

MOCK = {MOCK_MODELS!r}
MODEL_OWNER = "model_owner@example.com"
BENCHMARK_OWNER = "benchmark_owner@example.com"


def _dataset_dir(name):
    files = sy.resolve_dataset_files_path(name, owner_email=MODEL_OWNER)
    return str(files[0].parent)


# The model owner's guardrail: policy, threshold, declined message and the scoring code.
shield = sy.load_dataset_code("shieldgemma_model.shield_inference", owner_email=MODEL_OWNER)
print(f"Guardrail policy: '{{shield.POLICY_NAME}}'  threshold={{shield.THRESHOLD}}")


# ── Load the benchmark owner's private prompts ───────────────────────────────────────────
prompt_path = sy.resolve_dataset_file_path("safety_prompts", owner_email=BENCHMARK_OWNER)
with open(prompt_path, newline="") as f:
    prompt_rows = list(csv.DictReader(f))
print(f"Loaded {{len(prompt_rows)}} evaluation prompts")


# ── Stage 1: the model owner's guardrail ─────────────────────────────────────────────────
if MOCK:
    print("MOCK=True -- classifier replaced by a keyword heuristic, Gemma 3 by a placeholder")
    _DANGER_WORDS = ("weapon", "explosive", "bomb", "firearm", "poison", "kill", "terror", "exploit")
    _OUTPUT_WORDS = _DANGER_WORDS + ("hack",)

    def shield_score(prompt):
        hits = sum(w in prompt.lower() for w in _DANGER_WORDS)
        return min(1.0, 0.1 + 0.45 * hits)

    def shield_score_response(prompt, response):
        hits = sum(w in response.lower() for w in _OUTPUT_WORDS)
        return min(1.0, 0.1 + 0.45 * hits)

    def release_shield():
        pass
else:
    shield_dir = _dataset_dir("shieldgemma_model")
    print(f"Loading ShieldGemma from {{shield_dir}}...")
    clf = shield.setup(shield_dir)
    print("ShieldGemma loaded")

    def shield_score(prompt):
        return shield.score(clf, prompt)

    def shield_score_response(prompt, response):
        return shield.score_response(clf, prompt, response)

    def release_shield():
        shield.release(clf)


print(f"\\nStage 1 -- input gate: '{{shield.POLICY_NAME}}' (threshold {{shield.THRESHOLD}})")
results = []
for i, row in enumerate(prompt_rows):
    prompt = row["prompt_text"]
    score = shield_score(prompt)
    violates = shield.violates(score)
    verdict = "DECLINE" if violates else "allow"
    print(f"  [{{i+1}}/{{len(prompt_rows)}}] {{row['prompt_uid']}}  P(violation)={{score:.3f}}  {{verdict}}")
    results.append({{
        "prompt_id": row["prompt_uid"],
        "hazard": row.get("hazard"),
        "prompt": prompt,
        "shield_score": score,
        "violates_policy": violates,
        "completion": shield.DECLINED_MESSAGE if violates else None,
        "output_shield_score": None,
        "output_filtered": False,
        "ttft": None,
        "decode_tps": None,
    }})

to_answer = [r for r in results if not r["violates_policy"]]
print(f"\\n{{len(results) - len(to_answer)}} declined, {{len(to_answer)}} forwarded to Gemma 3")


# ── Stage 2: Gemma 3 on the allowed prompts, then the output gate ────────────────────────
# ShieldGemma stays loaded: every response is judged (in the context of its prompt) against
# the response-side wording of the same policy before it is written to the results.
if MOCK:
    def generate(prompt):
        # Echo the prompt so the output gate has something to judge in mock mode.
        return f"[mock completion for: {{prompt[:120]}}]", {{"ttft": 0.0, "decode_tps": 0.0}}
else:
    weights_dir = _dataset_dir("gemma3_model")
    gemma = sy.load_dataset_code("gemma3_model.gemma_inference", owner_email=MODEL_OWNER)
    print(f"\\nLoading Gemma 3 {MODEL_SIZE.upper()}-IT from {{weights_dir}}...")
    model, tokenizer, params = gemma.setup_model("{MODEL_SIZE}", weights_dir)
    print("Gemma 3 loaded")

    def generate(prompt):
        return gemma.generate(model, params, tokenizer, prompt, max_new_tokens=100)

print(f"\\nStage 2 -- Gemma 3 inference + output gate on {{len(to_answer)}} prompt(s)")
for i, r in enumerate(to_answer):
    print(f"  [{{i+1}}/{{len(to_answer)}}] {{r['prompt_id']}}: {{r['prompt'][:50]}}...")
    completion, stats = generate(r["prompt"])
    out_score = shield_score_response(r["prompt"], completion)
    filtered = shield.violates(out_score)
    print(f"      output gate: P(violation)={{out_score:.3f}}  {{'FILTER' if filtered else 'pass'}}")
    r["output_shield_score"] = out_score
    r["output_filtered"] = filtered
    # A filtered response is withheld entirely: only the notice reaches the results.
    r["completion"] = shield.OUTPUT_FILTERED_MESSAGE if filtered else completion
    r["ttft"] = stats["ttft"]
    r["decode_tps"] = stats["decode_tps"]
release_shield()

n_filtered = sum(1 for r in to_answer if r["output_filtered"])
n_answered = len(to_answer) - n_filtered


# ── Write outputs ────────────────────────────────────────────────────────────────────────
os.makedirs("outputs", exist_ok=True)
with open("outputs/safety_eval_results.json", "w") as f:
    json.dump({{
        "model": "mock" if MOCK else "{CKPT_SUBDIR}",
        "shield_model": "mock" if MOCK else "{SHIELD_REPO}",
        "policy": shield.POLICY_NAME,
        "shield_threshold": shield.THRESHOLD,
        "total_prompts": len(results),
        "declined": len(results) - len(to_answer),
        "output_filtered": n_filtered,
        "answered": n_answered,
        "results": results,
    }}, f, indent=2)

print(
    f"\\nDone. {{len(results)}} prompts: {{len(results) - len(to_answer)}} declined, "
    f"{{n_filtered}} output-filtered, {{n_answered}} answered."
)
'''

# Fail fast on a malformed job before it reaches the enclave.
compile(JOB_CODE, "main.py", "exec")
print(JOB_CODE)


## Step 10 — Benchmark owner submits the inference job

The job requests **both** of the model owner's datasets plus the benchmark. The dependency list is the
JAX stack for the Gemma 3 engine plus `torch` + `transformers` for ShieldGemma; with `MOCK_MODELS=True`
nothing is installed.


In [ ]:
GEMMA_DEPS = ["jax[cpu]", "flax", "orbax-checkpoint", "sentencepiece"]
SHIELD_DEPS = ["torch", "transformers"]
JOB_DEPS = [] if MOCK_MODELS else GEMMA_DEPS + SHIELD_DEPS

code_path = create_code_file(JOB_CODE)

benchmark_owner.submit_python_job(
    enclave.email,
    code_path,
    "safety_eval_job",
    datasets={
        model_owner.email: ["gemma3_model", "shieldgemma_model"],
        benchmark_owner.email: ["safety_prompts"],
    },
    share_results_with_do=False,
    dependencies=JOB_DEPS,
)

print(f"  Job 'safety_eval_job' submitted to enclave by {benchmark_owner.email}")
print(f"  Dependencies: {JOB_DEPS}")


## Step 11 — Enclave receives the job; both owners approve

The model owner reviews the job code before approving — it can read the public classifier code, the
policy text, and exactly how the two datasets are used.


In [ ]:
%%time
enclave.sync()
enclave.receive_jobs()
print(f"  Enclave job status : {enclave.jobs['safety_eval_job'].status}")

model_owner.sync()
benchmark_owner.sync()

model_owner.approve_job(model_owner.jobs["safety_eval_job"])
print("  Model owner approved")

benchmark_owner.approve_job(benchmark_owner.jobs["safety_eval_job"])
print("  Benchmark owner approved")

enclave.sync()
eval_status = enclave.jobs["safety_eval_job"].status
print(f"  Enclave job status: {eval_status}")
assert eval_status == "approved"
print("  Both approvals received — job is APPROVED")


## Step 12 — Enclave executes the job

With real models this is the slow step: it builds a venv with JAX + torch + transformers, loads
ShieldGemma (~5 GB), scores every prompt, then loads Gemma 3 and decodes the survivors.


In [ ]:
%%time
enclave.run_jobs()

eval_job = enclave.jobs["safety_eval_job"]
print(f"  Enclave job status: {eval_job.status}")
if eval_job.status != "done":
    print(eval_job.stderr)
assert eval_job.status == "done", f"job failed: {eval_job.status}"
print(eval_job.stdout)

enclave.distribute_results()
print("  Results distributed")


## Step 13 — Benchmark owner retrieves and inspects results

Declined prompts carry the declined message and no timing stats; answered prompts carry the Gemma 3
completion. Both carry the ShieldGemma score.


In [ ]:
benchmark_owner.sync()

eval_job = benchmark_owner.jobs["safety_eval_job"]
print(f"  Benchmark owner job status : {eval_job.status}")
assert eval_job.status == "done"
assert len(eval_job.output_paths) > 0

with open(eval_job.output_paths[0]) as f:
    result = json.load(f)

print()
print(f"  Model         : {result['model']}")
print(f"  Shield model  : {result['shield_model']}")
print(f"  Policy        : {result['policy']}  (threshold {result['shield_threshold']})")
print(
    f"  Total prompts : {result['total_prompts']}   declined={result['declined']}"
    f"   output_filtered={result['output_filtered']}   answered={result['answered']}"
)
print()
for r in result["results"]:
    completion = r["completion"]
    print(f"  prompt_id   : {r['prompt_id']}  [{r['hazard']}]")
    print(f"  prompt      : {r['prompt'][:120]}{'...' if len(r['prompt']) > 120 else ''}")
    in_verdict = "DECLINED" if r["violates_policy"] else "allowed"
    print(f"  input gate  : P(violation)={r['shield_score']:.3f}  → {in_verdict}")
    if not r["violates_policy"]:
        out_verdict = "FILTERED" if r["output_filtered"] else "passed"
        print(f"  output gate : P(violation)={r['output_shield_score']:.3f}  → {out_verdict}")
    print(f"  completion  : {completion[:120]}{'...' if len(completion) > 120 else ''}")
    if r["ttft"] is not None:
        print(f"  TTFT={r['ttft']:.2f}s  decode={r['decode_tps']:.1f} tok/s")
    print()


## Step 14 — Model owner is denied the results

`share_results_with_do=False`: the model owner contributed both models but only the submitter receives
the output.


In [ ]:
model_owner.sync()

mo_job = model_owner.jobs["safety_eval_job"]
mo_outputs = [p.name for p in mo_job.output_paths]
print(f"  Model owner — output files : {mo_outputs}")
assert len(mo_outputs) == 0, f"model owner must not receive job output, got {mo_outputs}"
print("  share_results_with_do=False — the model owner cannot see the job output")


---
## Summary

| Step | Actor | Action | Outcome |
|------|-------|--------|---------|
| 1 | Model owner | Upload `gemma3_model` (weights + engine) **and `shieldgemma_model` (weights + guardrail)** | Two private models in the enclave |
| 2 | Benchmark owner | Upload `safety_prompts` | Prompts available |
| 3 | Both | Share private data with enclave | Enclave can access all three assets |
| 4–7 | Model owner / both / enclave | `syft-restrict` job over the engine | Certificate + obfuscated engine → benchmark owner |
| 8–10 | Benchmark owner | Submit the **two-gate** inference job | Job sent to enclave |
| 11 | Both owners | `approve_job()` | Status → **approved** |
| 12 | Enclave | `run_jobs()` | **Input gate on every prompt; Gemma 3 answers the ones that pass; output gate on every response** |
| 13 | Benchmark owner | `sync()` + read output | Per-prompt scores + verdicts for both gates, and the completion or the notice that replaced it |

### What the gate does and does not give you

- The guardrail is the **model owner's**: policy, threshold and scoring code ship in `shield_inference.py`
  inside the ShieldGemma dataset. Changing the policy is a dataset update by the model owner, not a job
  change by the benchmark owner.
- The benchmark owner is not flying blind: the same file sits on the mock side, so it reads the exact
  policy before approving the job that enforces it.
- ShieldGemma runs **inside** the enclave on the private prompts, so the benchmark never leaves it to
  be classified.
- A filtered response is withheld entirely: the benchmark owner gets the notice and the score, never the
  text. That is the point of judging outputs too — a prompt can pass and the model can still say too much.
- The declined prompts still appear in the output with their score. If even the *existence* of a
  violating prompt must stay private, drop them from the results instead of annotating them.
- The guardrail is readable `transformers` code with nothing to hide, so it gets no restrict pass. What
  `syft-restrict` certifies is still only the Gemma 3 engine.
